In [ ]:
#connect with O-TFER and OSAM /done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module
'''
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att
'''

class Convv(nn.Module):
    # Standard convolution with args(ch_in, ch_out, kernel, stride, padding, groups, dilation, activation)
    default_act = nn.LeakyReLU()  # default activation

    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        if p is None:
            p = (k - 1) // 2 * d  # Calculate padding if not provided
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Branch processing layers
        self.avg_conv1 = Convv(1, 1, 3, 1)
        self.avg_conv2 = Convv(1, 1, 3, 1)
        self.avg_conv3 = Convv(1, 1, 3, 1)
        
        self.max_conv1 = Convv(1, 1, 3, 1)
        self.max_conv2 = Convv(1, 1, 3, 1)
        self.max_conv3 = Convv(1, 1, 3, 1)
        
        # Final attention convolution (without activation)
        self.att_conv = Convv(2, 1, kernel_size, 1, act=False)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        
        # Process average branch
        avg = self.avg_conv1(avg)
        avg = self.avg_conv2(avg)
        avg = self.avg_conv3(avg)
        
        # Process max branch
        mx = self.max_conv1(mx)
        mx = self.max_conv2(mx)
        mx = self.max_conv3(mx)
        
        # Combine and generate attention
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.att_conv(combined))
        return x * att

# TFER Classifier

import torch
import torch.nn as nn
import torch.nn.functional as F

class TFER(nn.Module):
    def __init__(self, input_size, num_classes=7):
        super().__init__()
        self.num_classes=num_classes
        
        # Step 1: First Decision (Neutral vs Non-neutral)
        self.neutral_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [neutral_score, non_neutral_score]
        )
        
        # Step 2: For Non-neutral only - Positive vs Negative
        self.valence_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [positive_score, negative_score]
        )
        
        # Step 3a: For Positive only - Happy vs Surprise
        self.positive_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [happy_score, surprise_score]
        )
        
        # Step 3b: For Negative only - Sad/Disgust/Anger/Fear
        self.negative_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # Output: [sad, disgust, anger, fear]
        )

    def forward(self, x):
        # Initialize output logits for all 7 classes
        batch_size = x.size(0)
        final_logits = torch.zeros(batch_size, 7).to(x.device)
        
        # --------------------------------------
        # STEP 1: Neutral vs Non-neutral
        # --------------------------------------
        neutral_logits = self.neutral_decision(x)
        neutral_probs = F.softmax(neutral_logits, dim=1)
        
        # Mark neutral samples (class 0)
        is_neutral = neutral_probs[:, 0] > 0.5
        final_logits[is_neutral, 0] = 10.0  # High score for neutral
        
        # Process non-neutral samples
        non_neutral = ~is_neutral
        non_neutral_x = x[non_neutral]
        
        if non_neutral_x.size(0) > 0:  # If any non-neutral samples
            # --------------------------------------
            # STEP 2: Positive vs Negative
            # --------------------------------------
            valence_logits = self.valence_decision(non_neutral_x)
            valence_probs = F.softmax(valence_logits, dim=1)
            
            is_positive = valence_probs[:, 0] > 0.5
            is_negative = ~is_positive
            
            # --------------------------------------
            # STEP 3a: Positive Branch - Happy vs Surprise
            # --------------------------------------
            if is_positive.any():
                positive_x = non_neutral_x[is_positive]
                pos_emotion_logits = self.positive_emotion(positive_x)
                final_logits[non_neutral][is_positive, 1:3] = pos_emotion_logits  # Classes 1-2
            
            # --------------------------------------
            # STEP 3b: Negative Branch - Sad/Disgust/Anger/Fear
            # --------------------------------------
            if is_negative.any():
                negative_x = non_neutral_x[is_negative]
                neg_emotion_logits = self.negative_emotion(negative_x)
                final_logits[non_neutral][is_negative, 3:7] = neg_emotion_logits  # Classes 3-6
        
        return final_logits


# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        try:
            train_model()
            #model.load_state_dict(torch.load(model_path))
            print("Loaded pretrained model")
        except RuntimeError as e:
            print(f"Error loading model: {e}")
            print("Training new model instead...")
            train_model()
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
#connect with O-TFER and spatial / done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att

'''
class Convv(nn.Module):
    # Standard convolution with args(ch_in, ch_out, kernel, stride, padding, groups, dilation, activation)
    default_act = nn.LeakyReLU()  # default activation

    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        if p is None:
            p = (k - 1) // 2 * d  # Calculate padding if not provided
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Branch processing layers
        self.avg_conv1 = Convv(1, 1, 3, 1)
        self.avg_conv2 = Convv(1, 1, 3, 1)
        self.avg_conv3 = Convv(1, 1, 3, 1)
        
        self.max_conv1 = Convv(1, 1, 3, 1)
        self.max_conv2 = Convv(1, 1, 3, 1)
        self.max_conv3 = Convv(1, 1, 3, 1)
        
        # Final attention convolution (without activation)
        self.att_conv = Convv(2, 1, kernel_size, 1, act=False)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        
        # Process average branch
        avg = self.avg_conv1(avg)
        avg = self.avg_conv2(avg)
        avg = self.avg_conv3(avg)
        
        # Process max branch
        mx = self.max_conv1(mx)
        mx = self.max_conv2(mx)
        mx = self.max_conv3(mx)
        
        # Combine and generate attention
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.att_conv(combined))
        return x * att
'''
# TFER Classifier

import torch
import torch.nn as nn
import torch.nn.functional as F

class TFER(nn.Module):
    def __init__(self, input_size, num_classes=7):
        super().__init__()
        self.num_classes=num_classes
        
        # Step 1: First Decision (Neutral vs Non-neutral)
        self.neutral_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [neutral_score, non_neutral_score]
        )
        
        # Step 2: For Non-neutral only - Positive vs Negative
        self.valence_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [positive_score, negative_score]
        )
        
        # Step 3a: For Positive only - Happy vs Surprise
        self.positive_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [happy_score, surprise_score]
        )
        
        # Step 3b: For Negative only - Sad/Disgust/Anger/Fear
        self.negative_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # Output: [sad, disgust, anger, fear]
        )

    def forward(self, x):
        # Initialize output logits for all 7 classes
        batch_size = x.size(0)
        final_logits = torch.zeros(batch_size, 7).to(x.device)
        
        # --------------------------------------
        # STEP 1: Neutral vs Non-neutral
        # --------------------------------------
        neutral_logits = self.neutral_decision(x)
        neutral_probs = F.softmax(neutral_logits, dim=1)
        
        # Mark neutral samples (class 0)
        is_neutral = neutral_probs[:, 0] > 0.5
        final_logits[is_neutral, 0] = 10.0  # High score for neutral
        
        # Process non-neutral samples
        non_neutral = ~is_neutral
        non_neutral_x = x[non_neutral]
        
        if non_neutral_x.size(0) > 0:  # If any non-neutral samples
            # --------------------------------------
            # STEP 2: Positive vs Negative
            # --------------------------------------
            valence_logits = self.valence_decision(non_neutral_x)
            valence_probs = F.softmax(valence_logits, dim=1)
            
            is_positive = valence_probs[:, 0] > 0.5
            is_negative = ~is_positive
            
            # --------------------------------------
            # STEP 3a: Positive Branch - Happy vs Surprise
            # --------------------------------------
            if is_positive.any():
                positive_x = non_neutral_x[is_positive]
                pos_emotion_logits = self.positive_emotion(positive_x)
                final_logits[non_neutral][is_positive, 1:3] = pos_emotion_logits  # Classes 1-2
            
            # --------------------------------------
            # STEP 3b: Negative Branch - Sad/Disgust/Anger/Fear
            # --------------------------------------
            if is_negative.any():
                negative_x = non_neutral_x[is_negative]
                neg_emotion_logits = self.negative_emotion(negative_x)
                final_logits[non_neutral][is_negative, 3:7] = neg_emotion_logits  # Classes 3-6
        
        return final_logits


# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        try:
            train_model()
            #model.load_state_dict(torch.load(model_path))
            print("Loaded pretrained model")
        except RuntimeError as e:
            print(f"Error loading model: {e}")
            print("Training new model instead...")
            train_model()
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
#connect with O-TFER wout spatial // done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module
'''
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att
'''

'''
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Branch processing layers
        self.avg_conv1 = Convv(1, 1, 3, 1)
        self.avg_conv2 = Convv(1, 1, 3, 1)
        self.avg_conv3 = Convv(1, 1, 3, 1)
        
        self.max_conv1 = Convv(1, 1, 3, 1)
        self.max_conv2 = Convv(1, 1, 3, 1)
        self.max_conv3 = Convv(1, 1, 3, 1)
        
        # Final attention convolution (without activation)
        self.att_conv = Convv(2, 1, kernel_size, 1, act=False)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        
        # Process average branch
        avg = self.avg_conv1(avg)
        avg = self.avg_conv2(avg)
        avg = self.avg_conv3(avg)
        
        # Process max branch
        mx = self.max_conv1(mx)
        mx = self.max_conv2(mx)
        mx = self.max_conv3(mx)
        
        # Combine and generate attention
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.att_conv(combined))
        return x * att
'''
# TFER Classifier

import torch
import torch.nn as nn
import torch.nn.functional as F

class TFER(nn.Module):
    def __init__(self, input_size, num_classes=7):
        super().__init__()
        self.num_classes=num_classes
        
        # Step 1: First Decision (Neutral vs Non-neutral)
        self.neutral_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [neutral_score, non_neutral_score]
        )
        
        # Step 2: For Non-neutral only - Positive vs Negative
        self.valence_decision = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [positive_score, negative_score]
        )
        
        # Step 3a: For Positive only - Happy vs Surprise
        self.positive_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # Output: [happy_score, surprise_score]
        )
        
        # Step 3b: For Negative only - Sad/Disgust/Anger/Fear
        self.negative_emotion = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # Output: [sad, disgust, anger, fear]
        )

    def forward(self, x):
        # Initialize output logits for all 7 classes
        batch_size = x.size(0)
        final_logits = torch.zeros(batch_size, 7).to(x.device)
        
        # --------------------------------------
        # STEP 1: Neutral vs Non-neutral
        # --------------------------------------
        neutral_logits = self.neutral_decision(x)
        neutral_probs = F.softmax(neutral_logits, dim=1)
        
        # Mark neutral samples (class 0)
        is_neutral = neutral_probs[:, 0] > 0.5
        final_logits[is_neutral, 0] = 10.0  # High score for neutral
        
        # Process non-neutral samples
        non_neutral = ~is_neutral
        non_neutral_x = x[non_neutral]
        
        if non_neutral_x.size(0) > 0:  # If any non-neutral samples
            # --------------------------------------
            # STEP 2: Positive vs Negative
            # --------------------------------------
            valence_logits = self.valence_decision(non_neutral_x)
            valence_probs = F.softmax(valence_logits, dim=1)
            
            is_positive = valence_probs[:, 0] > 0.5
            is_negative = ~is_positive
            
            # --------------------------------------
            # STEP 3a: Positive Branch - Happy vs Surprise
            # --------------------------------------
            if is_positive.any():
                positive_x = non_neutral_x[is_positive]
                pos_emotion_logits = self.positive_emotion(positive_x)
                final_logits[non_neutral][is_positive, 1:3] = pos_emotion_logits  # Classes 1-2
            
            # --------------------------------------
            # STEP 3b: Negative Branch - Sad/Disgust/Anger/Fear
            # --------------------------------------
            if is_negative.any():
                negative_x = non_neutral_x[is_negative]
                neg_emotion_logits = self.negative_emotion(negative_x)
                final_logits[non_neutral][is_negative, 3:7] = neg_emotion_logits  # Classes 3-6
        
        return final_logits


# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        #self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        #x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        try:
            train_model()
            #model.load_state_dict(torch.load(model_path))
            print("Loaded pretrained model")
        except RuntimeError as e:
            print(f"Error loading model: {e}")
            print("Training new model instead...")
            train_model()
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
#connect with s-TER w OSAM / done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module
'''
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att
'''


class Convv(nn.Module):
    # Standard convolution with args(ch_in, ch_out, kernel, stride, padding, groups, dilation, activation)
    default_act = nn.LeakyReLU()  # default activation

    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        if p is None:
            p = (k - 1) // 2 * d  # Calculate padding if not provided
        self.conv = nn.Conv2d(c1, c2, k, s, p, groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        # Branch processing layers
        self.avg_conv1 = Convv(1, 1, 3, 1)
        self.avg_conv2 = Convv(1, 1, 3, 1)
        self.avg_conv3 = Convv(1, 1, 3, 1)
        
        self.max_conv1 = Convv(1, 1, 3, 1)
        self.max_conv2 = Convv(1, 1, 3, 1)
        self.max_conv3 = Convv(1, 1, 3, 1)
        
        # Final attention convolution (without activation)
        self.att_conv = Convv(2, 1, kernel_size, 1, act=False)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        
        # Process average branch
        avg = self.avg_conv1(avg)
        avg = self.avg_conv2(avg)
        avg = self.avg_conv3(avg)
        
        # Process max branch
        mx = self.max_conv1(mx)
        mx = self.max_conv2(mx)
        mx = self.max_conv3(mx)
        
        # Combine and generate attention
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.att_conv(combined))
        return x * att


# TFER Classifier
class TFER(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.fc(x)

# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        #x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        train_model()
        #model.load_state_dict(torch.load(model_path))
        print("Loaded pretrained model")
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
#connect with s-TER w spa / done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att

# TFER Classifier
class TFER(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.fc(x)

# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        #x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        train_model()
        #model.load_state_dict(torch.load(model_path))
        print("Loaded pretrained model")
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
#connect with s-TER wout spa / done
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# Spatial Attention Module
'''
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2)
        
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg, mx], dim=1)
        att = torch.sigmoid(self.conv(combined))
        return x * att
'''
# TFER Classifier
class TFER(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        return self.fc(x)

# Branch Module
class Branch(nn.Module):
    def __init__(self, in_channels, spatial_size, num_classes):
        super().__init__()
        #self.attention = SpatialAttention()
        self.adaptive_pool = nn.AdaptiveAvgPool2d(spatial_size)
        self.flatten = nn.Flatten()
        self.classifier = TFER(in_channels * spatial_size * spatial_size, num_classes)
        
    def forward(self, x):
        #x = self.attention(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# VGG-16 with Branches
class VGG16Branches(nn.Module):
    def __init__(self, num_classes=7, spatial_size=7):
        super().__init__()
        self.branches = nn.ModuleList()
        self.num_classes = num_classes
        self.spatial_size = spatial_size
        
        # VGG-16 configuration
        cfg = [64, 64, 'M', 128, 128, 'M', 
               256, 256, 256, 'M', 
               512, 512, 512, 'M', 
               512, 512, 512, 'M']
        
        in_channels = 3
        layers = []
        conv_indices = []
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                layers.append(conv)
                conv_indices.append(len(layers)-1)
                layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                in_channels = v
        
        self.features = nn.ModuleList(layers)
        self.conv_indices = conv_indices
        
        # Create branches
        for pos in conv_indices:
            conv_layer = layers[pos]
            self.branches.append(Branch(conv_layer.out_channels, spatial_size, num_classes))
        
        # Main classifier
        self.adaptive_pool = nn.AdaptiveAvgPool2d((spatial_size, spatial_size))
        self.main_classifier = nn.Sequential(
            nn.Linear(512 * spatial_size * spatial_size, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )
        
    def forward(self, x):
        branch_outputs = []
        conv_count = 0
        
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.conv_indices:
                branch_out = self.branches[conv_count](x)
                branch_outputs.append(branch_out)
                conv_count += 1
                
        # Main stream
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        main_out = self.main_classifier(x)
        
        return branch_outputs, main_out

# Configuration
config = {
    'batch_size': 16,
    'epochs': 100,
    'data_path': 'D:/data/ck-all/all',  # Update with your dataset path
    'optimizer': 'Adam',
    'lr': 0.0001,
    'early_stop_patience': 100,
    'num_classes': 7,
    'log_dir': 'runs/vgg16_branches',
    'spatial_size': 7,
    'model_save_path': 'saved_models'
}

# Create directories
os.makedirs(config['model_save_path'], exist_ok=True)

# Data transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root=config['data_path'],
    transform=transform
)

# Split dataset
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config['batch_size'], shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config['batch_size']
)
test_loader = DataLoader(
    test_dataset, batch_size=config['batch_size']
)

# Initialize TensorBoard
writer = SummaryWriter(config['log_dir'])

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
model = VGG16Branches(
    num_classes=config['num_classes'],
    spatial_size=config['spatial_size']
).to(device)

# Log model graph
dummy_input = torch.randn(1, 3, 224, 224).to(device)
writer.add_graph(model, dummy_input)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'])

# Training function
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            branch_outs, main_out = model(inputs)
            
            loss_main = criterion(main_out, labels)
            loss_branches = sum(criterion(out, labels) for out in branch_outs)
            total_loss = loss_main + loss_branches
            
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                branch_outs, main_out = model(inputs)
                
                loss_main = criterion(main_out, labels)
                loss_branches = sum(criterion(out, labels) for out in branch_outs)
                val_loss += (loss_main + loss_branches).item()
        
        # Logging
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
            torch.save(model.state_dict(), model_path)
        else:
            patience_counter += 1
            if patience_counter >= config['early_stop_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

# Evaluation function
def evaluate_model(loader):
    model.eval()
    all_preds = {f'branch_{i}': [] for i in range(len(model.branches))}
    all_preds['main'] = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            branch_outs, main_out = model(inputs)
            
            for i, branch_out in enumerate(branch_outs):
                _, preds = torch.max(branch_out, 1)
                all_preds[f'branch_{i}'].extend(preds.cpu().numpy())
            
            _, main_preds = torch.max(main_out, 1)
            all_preds['main'].extend(main_preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return all_preds, np.array(all_labels)

# Main execution
if __name__ == "__main__":
    # Train or load model
    model_path = os.path.join(config['model_save_path'], 'best_vgg_model.pth')
    if os.path.exists(model_path):
        train_model()
        #model.load_state_dict(torch.load(model_path))
        print("Loaded pretrained model")
    else:
        print("Training new model...")
        train_model()
        model.load_state_dict(torch.load(model_path))
    
    # Evaluate
    print("\nEvaluating on test set...")
    test_preds, test_labels = evaluate_model(test_loader)
    
    # Calculate metrics
    results = {}
    for branch, preds in test_preds.items():
        preds_arr = np.array(preds)
        results[branch] = {
            'accuracy': (preds_arr == test_labels).mean(),
            'f1_score': f1_score(test_labels, preds_arr, average='macro'),
            'confusion_matrix': confusion_matrix(test_labels, preds_arr),
            'class_coverage': np.unique(preds_arr)
        }
    
    # Print results
    class_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    for branch, metrics in results.items():
        print(f"\n{branch}:")
        print(f"Accuracy: {metrics['accuracy']:.4f}")
        print(f"F1 Score: {metrics['f1_score']:.4f}")
        print(f"Class Coverage: {[class_labels[i] for i in metrics['class_coverage']]}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
    
    writer.close()

In [ ]:
# B-CNN / done
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import numpy as np
from PIL import Image
import os
import json
from collections import defaultdict
import cv2

# Fix for Windows DataLoader worker issue
import torch.multiprocessing as mp
mp.set_start_method('spawn', force=True)

# Define the hierarchical B-CNN model
class BranchCNN(nn.Module):
    def __init__(self, num_main_classes, num_sub_classes, pretrained=True):
        super(BranchCNN, self).__init__()
        
        # Load pre-trained VGG-16
        vgg = models.vgg16(pretrained=pretrained)
        
        # Shared feature extractor (up to certain layers)
        self.shared_features = nn.Sequential(*list(vgg.features.children())[:24])
        
        # Main branch (for main categories)
        self.main_branch = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Sub branch (for sub-categories)
        self.sub_branch = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Main classifier
        self.main_classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_main_classes)
        )
        
        # Sub classifier (takes both shared features and main branch context)
        self.sub_classifier = nn.Sequential(
            nn.Linear(512 + num_main_classes, 256),  # Concatenate main class info
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_sub_classes)
        )
        
        # Freeze early layers if desired
        for param in self.shared_features.parameters():
            param.requires_grad = False
        
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    
    def forward(self, x):
        # Normalize input
        x = self.normalize(x)
        
        # Shared feature extraction
        shared_features = self.shared_features(x)
        
        # Main branch
        main_features = self.main_branch(shared_features)
        main_output = self.main_classifier(main_features)
        
        # Sub branch with attention to main classification
        sub_features = self.sub_branch(shared_features)
        
        # Concatenate sub features with main class probabilities
        main_probs = F.softmax(main_output, dim=1)
        combined_features = torch.cat([sub_features, main_probs], dim=1)
        
        sub_output = self.sub_classifier(combined_features)
        
        return main_output, sub_output

# CK+ specific dataset class
class CKPlusDataset(Dataset):
    def __init__(self, root_dir, hierarchy_mapping, transform=None, is_train=True):
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train
        self.hierarchy_mapping = hierarchy_mapping
        
        # Load CK+ dataset structure
        self.samples = []
        self.classes = []
        self.class_to_idx = {}
        
        # CK+ typically has emotions as classes: anger, contempt, disgust, fear, happy, sadness, surprise
        emotion_folders = [f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))]
        emotion_folders.sort()
        
        self.classes = emotion_folders
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        # Collect all image samples
        for emotion in emotion_folders:
            emotion_path = os.path.join(root_dir, emotion)
            if os.path.isdir(emotion_path):
                image_files = [f for f in os.listdir(emotion_path) 
                             if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
                for img_file in image_files:
                    img_path = os.path.join(emotion_path, img_file)
                    self.samples.append((img_path, self.class_to_idx[emotion]))
        
        # Create main classes mapping (for hierarchical structure)
        # For CK+, we can group emotions into positive/negative or other categories
        self.main_classes = ['negative', 'positive']  # Example grouping
        self.main_to_idx = {main: idx for idx, main in enumerate(self.main_classes)}
        
        # Map emotions to main classes
        self.emotion_to_main = {
            'anger': 'negative',
            'contempt': 'negative',
            'disgust': 'negative',
            'fear': 'negative',
            'sadness': 'negative',
            'happy': 'positive',
            'surprise': 'positive'  # Could be considered positive or negative
        }
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
        print(f"Classes: {self.classes}")
        print(f"Main classes: {self.main_classes}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, sub_class_idx = self.samples[idx]
        sub_class_name = self.classes[sub_class_idx]
        main_class_name = self.emotion_to_main.get(sub_class_name, 'negative')
        main_class_idx = self.main_to_idx[main_class_name]
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            # If image loading fails, create a blank image
            image = Image.new('RGB', (224, 224), color='black')
        
        if self.transform:
            image = self.transform(image)
            
        return image, main_class_idx, sub_class_idx

# Hierarchical loss function
class HierarchicalLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super(HierarchicalLoss, self).__init__()
        self.alpha = alpha  # Weight for main vs sub loss
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, main_output, sub_output, main_target, sub_target):
        main_loss = self.ce_loss(main_output, main_target)
        sub_loss = self.ce_loss(sub_output, sub_target)
        
        # Combined loss with weighting
        total_loss = self.alpha * main_loss + (1 - self.alpha) * sub_loss
        return total_loss, main_loss, sub_loss

# Data augmentation for facial expressions
def get_transforms(input_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
    ])
    
    return train_transform, test_transform

# Training function with hierarchical evaluation
def train_hierarchical_model(model, train_loader, val_loader, criterion, optimizer, 
                           num_epochs=25, device='cuda'):
    model = model.to(device)
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        # Training phase
        model.train()
        running_loss = 0.0
        running_main_corrects = 0
        running_sub_corrects = 0
        
        for batch_idx, (inputs, main_labels, sub_labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            main_labels = main_labels.to(device)
            sub_labels = sub_labels.to(device)
            
            optimizer.zero_grad()
            
            main_output, sub_output = model(inputs)
            total_loss, main_loss, sub_loss = criterion(main_output, sub_output, 
                                                      main_labels, sub_labels)
            
            # Get predictions
            _, main_preds = torch.max(main_output, 1)
            _, sub_preds = torch.max(sub_output, 1)
            
            total_loss.backward()
            optimizer.step()
            
            running_loss += total_loss.item() * inputs.size(0)
            running_main_corrects += torch.sum(main_preds == main_labels.data)
            running_sub_corrects += torch.sum(sub_preds == sub_labels.data)
            
            # Print progress
            if batch_idx % 10 == 0:
                print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {total_loss.item():.4f}')
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_main_acc = running_main_corrects.double() / len(train_loader.dataset)
        epoch_sub_acc = running_sub_corrects.double() / len(train_loader.dataset)
        
        print(f'Train Loss: {epoch_loss:.4f} Main Acc: {epoch_main_acc:.4f} Sub Acc: {epoch_sub_acc:.4f}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_main_corrects = 0
        val_sub_corrects = 0
        
        with torch.no_grad():
            for inputs, main_labels, sub_labels in val_loader:
                inputs = inputs.to(device)
                main_labels = main_labels.to(device)
                sub_labels = sub_labels.to(device)
                
                main_output, sub_output = model(inputs)
                total_loss, main_loss, sub_loss = criterion(main_output, sub_output, 
                                                          main_labels, sub_labels)
                
                _, main_preds = torch.max(main_output, 1)
                _, sub_preds = torch.max(sub_output, 1)
                
                val_loss += total_loss.item() * inputs.size(0)
                val_main_corrects += torch.sum(main_preds == main_labels.data)
                val_sub_corrects += torch.sum(sub_preds == sub_labels.data)
        
        val_loss = val_loss / len(val_loader.dataset)
        val_main_acc = val_main_corrects.double() / len(val_loader.dataset)
        val_sub_acc = val_sub_corrects.double() / len(val_loader.dataset)
        
        print(f'Val Loss: {val_loss:.4f} Main Acc: {val_main_acc:.4f} Sub Acc: {val_sub_acc:.4f}')
        
        # Save best model
        current_acc = (val_main_acc + val_sub_acc) / 2
        if current_acc > best_acc:
            best_acc = current_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': val_loss,
                'main_acc': val_main_acc,
                'sub_acc': val_sub_acc
            }, 'best_branch_cnn_model.pth')
            print(f'New best model saved with accuracy: {best_acc:.4f}')
        
        print()
    
    print(f'Best combined Acc: {best_acc:.4f}')
    return model

# Main function
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    # CK+ dataset path - adjust this to your CK+ dataset location
    ckplus_root = 'D:/data/ck-all/all'
    
    # Create hierarchy mapping for CK+
    hierarchy_mapping = {
        'anger': 'negative',
        'contempt': 'negative',
        'disgust': 'negative',
        'fear': 'negative',
        'sadness': 'negative',
        'happy': 'positive',
        'surprise': 'positive'
    }
    
    # Create datasets
    train_transform, test_transform = get_transforms()
    
    # For CK+, you might want to use cross-validation instead of train/val split
    # or use a predefined split if available
    train_dataset = CKPlusDataset(
        ckplus_root, hierarchy_mapping, transform=train_transform, is_train=True
    )
    
    # For simplicity, using the same dataset for validation
    # In practice, you should split your data properly
    val_dataset = CKPlusDataset(
        ckplus_root, hierarchy_mapping, transform=test_transform, is_train=False
    )
    
    print(f'Number of main classes: {len(train_dataset.main_classes)}')
    print(f'Number of sub classes: {len(train_dataset.classes)}')
    
    # FIX: Set num_workers to 0 for Windows compatibility
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    
    # Initialize model
    model = BranchCNN(
        num_main_classes=len(train_dataset.main_classes),
        num_sub_classes=len(train_dataset.classes),
        pretrained=True
    )
    
    # Define loss and optimizer
    criterion = HierarchicalLoss(alpha=0.6)
    optimizer = torch.optim.Adam([
        {'params': model.main_branch.parameters()},
        {'params': model.sub_branch.parameters()},
        {'params': model.main_classifier.parameters()},
        {'params': model.sub_classifier.parameters()}
    ], lr=0.0001, weight_decay=1e-4)
    
    # Train model
    try:
        model = train_hierarchical_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            num_epochs=20,  # Reduced for testing
            device=device
        )
        
        # Save final model
        torch.save(model.state_dict(), 'final_branch_cnn_model.pth')
        print('Training completed successfully!')
        
    except Exception as e:
        print(f"Error during training: {e}")
        print("Trying with single process...")
        
        # Fallback: try with even simpler settings
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=False)
        val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=False)
        
        # Try training again
        model = train_hierarchical_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            num_epochs=10,
            device=device
        )

# Alternative simple dataset loader without multiprocessing issues
class SimpleCKPlusLoader:
    def __init__(self, root_dir, batch_size=4):
        self.dataset = CKPlusDataset(root_dir, {}, transforms.ToTensor())
        self.batch_size = batch_size
        
    def __iter__(self):
        self.current_idx = 0
        return self
    
    def __next__(self):
        if self.current_idx >= len(self.dataset):
            raise StopIteration
        
        batch_images = []
        batch_main_labels = []
        batch_sub_labels = []
        
        for _ in range(self.batch_size):
            if self.current_idx >= len(self.dataset):
                break
            img, main_label, sub_label = self.dataset[self.current_idx]
            batch_images.append(img)
            batch_main_labels.append(main_label)
            batch_sub_labels.append(sub_label)
            self.current_idx += 1
        
        if not batch_images:
            raise StopIteration
            
        return (torch.stack(batch_images), 
                torch.tensor(batch_main_labels), 
                torch.tensor(batch_sub_labels))

if __name__ == '__main__':
    main()

In [ ]:
# basiline vgg16 code / done
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
from sklearn.metrics import f1_score
# Function to load data from train, valid, test folders
def load_data(data_dir, batch_size):
    transform = transforms.Compose([
    transforms.Resize((48, 48)),
    #transforms.RandomHorizontalFlip(),
    #transforms.RandomRotation(10),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

    train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
    valid_dataset = datasets.ImageFolder(os.path.join(data_dir, 'valid'), transform=transform)
    test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, valid_loader, test_loader

# Function to define and train VGG16
def train_vgg166(data_dir, num_classes, batch_size=16, num_epochs=10, learning_rate=0.001):
    # Load data
    train_loader, valid_loader, test_loader = load_data(data_dir, batch_size)

    # Load VGG16 model
    model = models.vgg16(pretrained=False)
    model.classifier[6] = nn.Linear(4096, num_classes)  # Replace final layer with the number of classes

    # Define loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Check if GPU is available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Validation step
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
    
                

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, "
              f"Validation Loss: {val_loss/len(valid_loader):.4f}, "
              f"Validation Accuracy: {100 * correct / total:.2f}%")
    test_accuracy = 100 * correct / total
        

    # Save the trained model
    torch.save(model.state_dict(), "vgg16_trained_weights.pth")
    print("Model training complete and saved as 'vgg16_trained_weights.pth'")

    # Test the model
    '''
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    #test_accuracy = 100 * correct / total
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    '''

    return model, test_accuracy

from sklearn.utils.class_weight import compute_class_weight
import numpy as np


def train_vgg16(data_dir, num_classes, batch_size=16, num_epochs=30, learning_rate=0.001,patience=100):
    train_loader, valid_loader, test_loader = load_data(data_dir, batch_size)

    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(4096, num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Compute class weights
    train_labels = [label for _, label in train_loader.dataset]
    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)  # Weighted loss
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    

    best_val_loss = float('inf')
    accumulation_steps = 4  # Gradient accumulation steps
    patience_counter=0

    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': [], 'f1_scores': []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()  # Initialize gradients

        for step, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels) / accumulation_steps  # Scale loss
            loss.backward()  # Accumulate gradients

            # Perform optimization step every 'accumulation_steps'
            if (step + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()  # Clear gradients after step

            running_loss += loss.item() * accumulation_steps  # Multiply back for accurate logging

        # Final optimization step for remaining gradients (if any)
        if (step + 1) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        # Validation step
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        #test_accuracy = 100 * correct / total

        

        val_accuracy = 100 * correct / total
        #scheduler.step(val_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, "
              f"Validation Loss: {val_loss/len(valid_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")
        

        train_loss = running_loss / len(train_loader)
        history['train_loss'].append(train_loss)
    
        f1 = calculate_f1_score_per_epoch(model, valid_loader)\
        
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['f1_scores'].append(f1)

        print(f"F1-Score: {f1:.4f}")

        # Log validation loss and accuracy
        #writer.add_scalar("Loss-avg/Validation", avg_valid_loss, epoch)
        #writer.add_scalar("Accuracy-avg/Validation", valid_accuracy, epoch)
        
        '''
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_vgg16_model.pth")
        '''      
        #Early

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save the best model
            torch.save(model.state_dict(), "best_vgg16_model.pth")
            print(f"Best model saved at Epoch {epoch+1} with Validation Loss: {val_loss/len(valid_loader):.4f}")
        else:
            patience_counter += 1
            print(f"Patience counter: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered. Stopping training.")
                break
    print("History content:", history)
    plot_training_curves(history)
    plot_f1_score_curve(history['f1_scores'])


    model.load_state_dict(torch.load("best_vgg16_model.pth",weights_only=True))


    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_accuracy = 100 * correct / total
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    

    return model, test_accuracy

#######
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from torchvision.models import VGG16_Weights
def train_vgg167(data_dir, num_classes, batch_size=16, num_epochs=50, learning_rate=0.001, patience=5):
    train_loader, valid_loader, test_loader = load_data(data_dir, batch_size)

    model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    model.classifier[6] = nn.Linear(4096, num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    # Compute class weights
    train_labels = [label for _, label in train_loader.dataset]
    class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    

    # Early stopping variables
    best_val_loss = float('inf')
    patience_counter = 0
    accumulation_steps = 4  # Gradient accumulation steps

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()  # Initialize gradients

        for step, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels) / accumulation_steps  # Scale loss
            loss.backward()  # Accumulate gradients

            # Perform optimization step every 'accumulation_steps'
            if (step + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()  # Clear gradients after step

            running_loss += loss.item() * accumulation_steps  # Multiply back for accurate logging

        # Final optimization step for remaining gradients (if any)
        if (step + 1) % accumulation_steps != 0:
            optimizer.step()
            optimizer.zero_grad()

        # Validation step
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        #scheduler.step(val_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, "
              f"Validation Loss: {val_loss/len(valid_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")

        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_vgg16_model.pth")
            print(f"Best model saved at Epoch {epoch+1} with Validation Loss: {val_loss/len(valid_loader):.4f}")
        else:
            patience_counter += 1
            print(f"Patience counter: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered. Stopping training.")
                break

    # Load the best model weights
    model.load_state_dict(torch.load("best_vgg16_model.pth", weights_only=True))
    return model

def evaluate_model(model, data_loader, phase):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    correct = 0
    total = 0
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(data_loader)

    print(f"{phase} Accuracy: {accuracy:.2f}%, Loss: {avg_loss:.4f}")
    return accuracy, avg_loss  

import matplotlib.pyplot as plt
def plot_training_curves(history):

    if not history['train_loss'] or not history['val_loss']:
        print("No training data to plot. Ensure training ran successfully.")
        return
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot training and validation loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()

    # Plot validation accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()
def plot_f1_score_curve(f1_scores):
    """Plot F1-score curve."""
    epochs = range(1, len(f1_scores) + 1)
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, f1_scores, label='F1-Score', marker='o')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.title('F1-Score Curve')
    plt.legend()
    plt.grid()
    plt.show()
    
import matplotlib.pyplot as plt
def calculate_f1_score_per_epoch(model, data_loader):
    """Calculate F1-score for each epoch."""
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    # Calculate F1-score
    f1 = f1_score(y_true, y_pred, average='weighted')
    return f1
###
# Example usage


data_dir = "D:/data/ck-vgg16"
num_classes = 7
batch_size = 16
num_epochs = 100
learning_rate = 0.0001
patience = 5 # Stop training if no improvement for 5 epochs

trained_model = train_vgg16(data_dir, num_classes, batch_size, num_epochs, learning_rate,patience)

